# Little Steps — Revised Interactive Version

**Important:** Keep the original uploaded notebook as your backup. This revised copy preserves the project work and adds a safer, upgraded app launcher at the end.

- Do **not** use **Runtime → Run all**. This notebook contains setup steps, manual file-upload cells, and an experimental ML section that are not intended to run as one uninterrupted sequence.
- The two old database-reset repair cells have been disabled in this revised copy because they could erase saved image/audio paths and completion progress.
- The older separate interface/dashboard/welcome cells are marked as superseded. Use the **FINAL — Launch Little Steps Interactive App** cell at the end for the unified interface.
- The final launcher reads your existing `MyDrive/LittleSteps` database and media folders. It does not delete or recreate your activities. Completion changes are saved back to Google Drive.


In [ ]:
# Install the libraries for Little Steps

!pip install -q openai pandas matplotlib pillow scikit-learn

In [ ]:
# LITTLE STEPS: Activity Dataset

activities = [
    {
        "activity_id": 1,
        "name": "Washing Hands",
        "category": "Hygiene",
        "steps": [
            "Turn on the water.",
            "Wet your hands.",
            "Apply soap.",
            "Rub your hands together.",
            "Rinse your hands.",
            "Dry your hands."
        ]
    },

    {
        "activity_id": 2,
        "name": "Brushing Teeth",
        "category": "Hygiene",
        "steps": [
            "Pick up your toothbrush.",
            "Put toothpaste on the brush.",
            "Brush your teeth gently.",
            "Spit out the toothpaste.",
            "Rinse your mouth."
        ]
    },

    {
        "activity_id": 3,
        "name": "Tying Shoes",
        "category": "Dressing",
        "steps": [
            "Cross the two laces.",
            "Pull one lace underneath.",
            "Make a loop.",
            "Wrap the other lace around.",
            "Pull it through to make a bow."
        ]
    }
]

# Display our activities using a loop

for activity in activities:
    print("\nActivity:", activity["name"])

    for number, step in enumerate(activity["steps"], start=1):
        print(number, "-", step)


Activity: Washing Hands
1 - Turn on the water.
2 - Wet your hands.
3 - Apply soap.
4 - Rub your hands together.
5 - Rinse your hands.
6 - Dry your hands.

Activity: Brushing Teeth
1 - Pick up your toothbrush.
2 - Put toothpaste on the brush.
3 - Brush your teeth gently.
4 - Spit out the toothpaste.
5 - Rinse your mouth.

Activity: Tying Shoes
1 - Cross the two laces.
2 - Pull one lace underneath.
3 - Make a loop.
4 - Wrap the other lace around.
5 - Pull it through to make a bow.


In [ ]:
import pandas as pd

# Convert our activity list into a table

activity_summary = []

for activity in activities:
    activity_summary.append({
        "Activity": activity["name"],
        "Category": activity["category"],
        "Number of Steps": len(activity["steps"])
    })

df = pd.DataFrame(activity_summary)

# Display our dataset
display(df)

,Activity,Category,Number of Steps
0,Washing Hands,Hygiene,6
1,Brushing Teeth,Hygiene,5
2,Tying Shoes,Dressing,5


In [ ]:
import sqlite3

# Create a database
connection = sqlite3.connect("little_steps.db")

# Create a table
cursor = connection.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS activity_steps (
    activity_id INTEGER,
    activity_name TEXT,
    category TEXT,
    step_number INTEGER,
    instruction TEXT,
    image_path TEXT,
    audio_path TEXT
)
""")

# Add our activity instructions
cursor.execute("DELETE FROM activity_steps")

for activity in activities:
    for number, step in enumerate(activity["steps"], start=1):

        cursor.execute("""
        INSERT INTO activity_steps
        VALUES (?, ?, ?, ?, ?, ?, ?)
        """, (
            activity["activity_id"],
            activity["name"],
            activity["category"],
            number,
            step,
            None,
            None
        ))

connection.commit()

print("Little Steps database created successfully!")

OperationalError: table activity_steps has 8 columns but 7 values were supplied

In [ ]:
cursor.execute("""
SELECT activity_name, step_number, instruction
FROM activity_steps
ORDER BY activity_id, step_number
""")

saved_steps = cursor.fetchall()

for activity_name, number, instruction in saved_steps:
    print(f"{activity_name} | Step {number}: {instruction}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import shutil

database_folder = "/content/drive/MyDrive/LittleSteps"
os.makedirs(database_folder, exist_ok=True)

shutil.copy(
    "little_steps.db",
    database_folder + "/little_steps.db"
)

print("Database saved to Google Drive!")

In [ ]:
import os

base_folder = "/content/drive/MyDrive/LittleSteps"

image_folder = os.path.join(base_folder, "images")
audio_folder = os.path.join(base_folder, "audio")

os.makedirs(image_folder, exist_ok=True)
os.makedirs(audio_folder, exist_ok=True)

print("Image folder:", image_folder)
print("Audio folder:", audio_folder)
print("Folders created successfully!")

In [ ]:
!pip install -q pillow

In [ ]:
from PIL import Image, ImageDraw

# Create a blank illustration
img = Image.new("RGB", (600, 400), "#EAF7FF")
draw = ImageDraw.Draw(img)

# Draw a faucet
draw.rectangle((230, 100, 370, 140), fill="#A0A0A0")
draw.rectangle((280, 140, 320, 190), fill="#A0A0A0")

# Draw water
draw.line((300, 190, 300, 270), fill="#2196F3", width=8)

# Draw sink
draw.rounded_rectangle(
    (150, 270, 450, 340),
    radius=35,
    fill="#B3E5FC",
    outline="#4682B4",
    width=5
)

# Save image
image_path = os.path.join(
    image_folder,
    "washing_hands_step_1.png"
)

img.save(image_path)

display(img)

print("Illustration saved!")

In [ ]:
!pip install -q edge-tts

In [ ]:
import asyncio
import edge_tts
import os
from IPython.display import Audio, display

async def generate_audio():
    text = "Turn on the water."

    voice = "en-US-AriaNeural"

    audio_path = os.path.join(
        audio_folder,
        "washing_hands_step_1.mp3"
    )

    communicate = edge_tts.Communicate(text, voice)
    await communicate.save(audio_path)

    return audio_path

audio_path = await generate_audio()

display(Audio(audio_path))
print("Audio generated successfully!")

In [ ]:
cursor.execute("""
UPDATE activity_steps
SET audio_path = ?
WHERE activity_id = 1 AND step_number = 1
""", (audio_path,))

connection.commit()

shutil.copy(
    "little_steps.db",
    database_folder + "/little_steps.db"
)

print("Audio connected to database!")

In [ ]:
!pip install -q pillow

In [ ]:
from PIL import Image, ImageDraw, ImageFont
import os
import textwrap

def create_step_image(activity, step, number):

    width, height = 700, 500
    img = Image.new("RGB", (width, height), "#F0F8FF")
    draw = ImageDraw.Draw(img)

    # Activity colors
    colors = {
        "Washing Hands": "#81C784",
        "Brushing Teeth": "#64B5F6",
        "Tying Shoes": "#FFB74D"
    }

    color = colors[activity["name"]]

    # Background design
    draw.rounded_rectangle(
        (35, 35, 665, 465),
        radius=30,
        fill="white",
        outline=color,
        width=8
    )

    # Step number
    draw.ellipse((65, 65, 145, 145), fill=color)
    draw.text((90, 88), str(number), fill="white")

    # Activity title
    draw.text(
        (175, 90),
        activity["name"],
        fill="#263238"
    )

    # Simple activity illustration
    if activity["name"] == "Washing Hands":
        draw.rectangle((270, 190, 430, 225), fill="#9E9E9E")
        draw.rectangle((320, 225, 380, 280), fill="#9E9E9E")
        draw.line((350, 280, 350, 340), fill="#2196F3", width=12)
        draw.ellipse((310, 330, 390, 380), fill="#90CAF9")

    elif activity["name"] == "Brushing Teeth":
        draw.rounded_rectangle(
            (320, 195, 370, 355),
            radius=20,
            fill="#64B5F6"
        )
        draw.rectangle((305, 180, 385, 210), fill="#90A4AE")
        for x in range(315, 385, 15):
            draw.line((x, 180, x, 160), fill="#263238", width=4)

    else:
        draw.ellipse((240, 240, 460, 350), fill="#FFCC80")
        draw.rectangle((330, 200, 390, 300), fill="#FFCC80")
        draw.line((330, 220, 280, 170), fill="#795548", width=9)
        draw.line((390, 220, 440, 170), fill="#795548", width=9)

    # Instruction text
    wrapped = textwrap.fill(step, width=38)

    draw.multiline_text(
        (100, 395),
        wrapped,
        fill="#263238",
        spacing=8
    )

    filename = (
        activity["name"].lower().replace(" ", "_")
        + f"_step_{number}.png"
    )

    path = os.path.join(image_folder, filename)
    img.save(path)

    return path

In [ ]:
for activity in activities:

    for number, step in enumerate(activity["steps"], start=1):

        path = create_step_image(
            activity,
            step,
            number
        )

        cursor.execute("""
        UPDATE activity_steps
        SET image_path = ?
        WHERE activity_id = ? AND step_number = ?
        """, (
            path,
            activity["activity_id"],
            number
        ))

connection.commit()

print("All activity illustrations generated!")

In [ ]:
from IPython.display import display

for activity in activities:

    path = os.path.join(
        image_folder,
        activity["name"].lower().replace(" ", "_")
        + "_step_1.png"
    )

    print(activity["name"])
    display(Image.open(path))

In [ ]:
import shutil

shutil.copy(
    "little_steps.db",
    database_folder + "/little_steps.db"
)

print("All images and audio paths saved!")

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import os
import shutil

filename = next(iter(uploaded))

image_path = os.path.join(
    image_folder,
    "washing_hands_step_1.png"
)

shutil.move(filename, image_path)

print("3D illustration saved!")

In [ ]:
cursor.execute("""
UPDATE activity_steps
SET image_path = ?
WHERE activity_id = 1 AND step_number = 1
""", (image_path,))

connection.commit()

shutil.copy(
    "little_steps.db",
    database_folder + "/little_steps.db"
)

print("Database updated with 3D illustration!")

In [ ]:
from IPython.display import display, Image

display(Image(filename=image_path))

In [ ]:
from google.colab import files

uploaded = files.upload()

sheet_path = "little_steps_sheet.png"

In [ ]:
from PIL import Image
import os

sheet = Image.open(sheet_path).convert("RGB")

print("Poster dimensions:", sheet.size)

# Crop boundaries for the four instructional panels
x_ranges = [
    (230, 532),
    (536, 843),
    (846, 1155),
    (1158, 1525)
]

y_ranges = [
    (10, 337),
    (350, 675),
    (690, 1015)
]

activity_names = [
    "washing_hands",
    "brushing_teeth",
    "tying_shoes"
]

for row, activity in enumerate(activity_names):

    for col in range(4):

        image = sheet.crop((
            x_ranges[col][0],
            y_ranges[row][0],
            x_ranges[col][1],
            y_ranges[row][1]
        ))

        filename = f"{activity}_step_{col+1}.png"

        save_path = os.path.join(
            image_folder,
            filename
        )

        image.save(save_path)

print("12 individual illustrations saved!")

In [ ]:
from PIL import Image
import os
from IPython.display import display

sheet = Image.open("little_steps_sheet.png").convert("RGB")

# Exact boundaries for your uploaded poster
x_bounds = [
    (7, 381),
    (390, 764),
    (773, 1146),
    (1155, 1529)
]

y_bounds = [
    (8, 341),
    (350, 665),
    (674, 1016)
]

filenames = [
    [
        "washing_hands_cover.png",
        "washing_hands_step_1.png",
        "washing_hands_step_3.png",
        "washing_hands_step_4.png"
    ],
    [
        "brushing_teeth_cover.png",
        "brushing_teeth_step_2.png",
        "brushing_teeth_step_3.png",
        "brushing_teeth_step_4.png"
    ],
    [
        "tying_shoes_cover.png",
        "tying_shoes_step_2.png",
        "tying_shoes_step_3.png",
        "tying_shoes_step_4.png"
    ]
]

os.makedirs(image_folder, exist_ok=True)

for row in range(3):
    for col in range(4):

        left, right = x_bounds[col]
        top, bottom = y_bounds[row]

        cropped = sheet.crop((left, top, right, bottom))

        save_path = os.path.join(
            image_folder,
            filenames[row][col]
        )

        cropped.save(save_path)

        print("Saved:", filenames[row][col])
        display(cropped)

print("All 12 illustrations cropped successfully!")

In [ ]:
for filename in os.listdir(image_folder):
    if filename.endswith(".png"):
        print(filename)

In [ ]:
import os
import shutil
import sqlite3

# Connect to your database
connection = sqlite3.connect("little_steps.db")
cursor = connection.cursor()

# Match image files to database activity IDs and step numbers
image_mapping = {
    # Washing hands
    (1, 1): "washing_hands_cover.png",
    (1, 2): "washing_hands_step_1.png",
    (1, 3): "washing_hands_step_3.png",
    (1, 4): "washing_hands_step_4.png",
    (1, 5): "washing_hands_step_4.png",

    # Brushing teeth
    (2, 1): "brushing_teeth_cover.png",
    (2, 2): "brushing_teeth_step_2.png",
    (2, 3): "brushing_teeth_step_3.png",
    (2, 5): "brushing_teeth_step_4.png",

    # Tying shoes
    (3, 1): "tying_shoes_cover.png",
    (3, 3): "tying_shoes_step_2.png",
    (3, 4): "tying_shoes_step_3.png",
    (3, 5): "tying_shoes_step_4.png"
}

for (activity_id, step_number), filename in image_mapping.items():

    path = os.path.join(image_folder, filename)

    if os.path.exists(path):
        cursor.execute("""
            UPDATE activity_steps
            SET image_path = ?
            WHERE activity_id = ? AND step_number = ?
        """, (path, activity_id, step_number))

        print("Connected:", filename)

    else:
        print("Missing image:", filename)

connection.commit()

print("Database image connections updated!")

In [ ]:
database_folder = "/content/drive/MyDrive/LittleSteps"

# Save the updated database
shutil.copy(
    "little_steps.db",
    os.path.join(database_folder, "little_steps.db")
)

print("Updated database saved!")

In [ ]:
cursor.execute("""
SELECT activity_name, step_number, instruction, image_path
FROM activity_steps
ORDER BY activity_id, step_number
""")

for activity, number, instruction, path in cursor.fetchall():
    status = "Image available" if path and os.path.exists(path) else "No image"
    print(f"{activity} | Step {number}: {instruction} | {status}")

In [ ]:
!pip install -q edge-tts

In [ ]:
import edge_tts
import asyncio
import os
import shutil
import sqlite3

# Reconnect to database
connection = sqlite3.connect("little_steps.db")
cursor = connection.cursor()

async def generate_all_audio():

    cursor.execute("""
        SELECT activity_id, activity_name,
               step_number, instruction
        FROM activity_steps
        ORDER BY activity_id, step_number
    """)

    rows = cursor.fetchall()

    successful = 0
    failed = []

    for activity_id, activity_name, number, instruction in rows:

        filename = (
            activity_name.lower().replace(" ", "_")
            + f"_step_{number}.mp3"
        )

        path = os.path.join(audio_folder, filename)

        try:
            speech = edge_tts.Communicate(
                instruction,
                voice="en-US-AriaNeural"
            )

            await speech.save(path)

            cursor.execute("""
                UPDATE activity_steps
                SET audio_path = ?
                WHERE activity_id = ? AND step_number = ?
            """, (path, activity_id, number))

            successful += 1
            print("Created:", filename)

        except Exception as error:
            failed.append((filename, str(error)))
            print("Failed:", filename, error)

    connection.commit()

    # Save updated database to Drive
    shutil.copy(
        "little_steps.db",
        os.path.join(database_folder, "little_steps.db")
    )

    print("\nAudio generation finished!")
    print("Successful:", successful)
    print("Failed:", len(failed))

    return failed

failed_audio = await generate_all_audio()

In [ ]:
fcursor.execute("""
SELECT activity_name, step_number, audio_path
FROM activity_steps
ORDER BY activity_id, step_number
""")

for activity, number, path in cursor.fetchall():

    available = (
        path is not None
        and os.path.exists(path)
    )

    print(
        activity,
        "| Step:", number,
        "| Audio:", "AVAILABLE" if available else "MISSING"
    )

In [ ]:
from IPython.display import Audio, display

cursor.execute("""
SELECT audio_path
FROM activity_steps
WHERE activity_id = 2 AND step_number = 2
""")

path = cursor.fetchone()[0]

display(Audio(filename=path))

In [ ]:
import asyncio
import edge_tts
import os
import sqlite3
import shutil

connection = sqlite3.connect("little_steps.db")
cursor = connection.cursor()

async def generate_everything():

    cursor.execute("""
        SELECT activity_id, activity_name, step_number, instruction
        FROM activity_steps
        ORDER BY activity_id, step_number
    """)

    records = cursor.fetchall()
    completed = 0

    for activity_id, activity, number, instruction in records:

        filename = (
            activity.lower().replace(" ", "_")
            + f"_step_{number}.mp3"
        )

        path = os.path.join(audio_folder, filename)

        try:
            communicate = edge_tts.Communicate(
                instruction,
                voice="en-US-AriaNeural"
            )

            await communicate.save(path)

            cursor.execute("""
                UPDATE activity_steps
                SET audio_path = ?
                WHERE activity_id = ? AND step_number = ?
            """, (path, activity_id, number))

            completed += 1
            print(f"✓ {activity} — Step {number}")

        except Exception as error:
            print(f"✗ {activity} — Step {number}: {error}")

    connection.commit()

    shutil.copy(
        "little_steps.db",
        os.path.join(database_folder, "little_steps.db")
    )

    print(f"\nFinished! {completed}/{len(records)} audio files generated.")

await generate_everything()

In [ ]:
cursor.execute("""
SELECT activity_name,
       COUNT(*) AS total_steps,
       SUM(
           CASE WHEN audio_path IS NOT NULL
           AND audio_path != ''
           AND audio_path != 'None'
           AND audio_path != 'NULL'
           THEN 1 ELSE 0 END
       ) AS audio_paths
FROM activity_steps
GROUP BY activity_name
""")

for row in cursor.fetchall():
    print(row)

In [ ]:
import sqlite3
import os

connection = sqlite3.connect("little_steps.db")
cursor = connection.cursor()

# Add completion column if it doesn't already exist
cursor.execute("PRAGMA table_info(activity_steps)")
columns = [column[1] for column in cursor.fetchall()]

if "completed" not in columns:
    cursor.execute("""
    ALTER TABLE activity_steps
    ADD COLUMN completed INTEGER DEFAULT 0
    """)

connection.commit()

print("Progress tracking added!")

In [ ]:
import os

for file in os.listdir(image_folder):
    if "washing_hands" in file:
        print(file)

### Older standalone activity viewer — superseded

This older interface is preserved in the original uploaded notebook. In this revised copy, use the unified interactive app launcher in the final section instead, to avoid duplicate interfaces.

In [ ]:
import os
import sqlite3
import shutil

connection = sqlite3.connect("little_steps.db")
cursor = connection.cursor()

# Assign the correct image to every washing-hands step
for step in range(1, 7):
    filename = f"washing_hands_step_{step}.png"
    image_path = os.path.join(image_folder, filename)

    cursor.execute("""
        UPDATE activity_steps
        SET image_path = ?
        WHERE activity_id = 1 AND step_number = ?
    """, (image_path, step))

connection.commit()

# Save updated database to Google Drive
shutil.copy(
    "little_steps.db",
    os.path.join(database_folder, "little_steps.db")
)

print("All 6 washing-hands images updated!")

In [ ]:
import sqlite3
import os

connection = sqlite3.connect("little_steps.db")
cursor = connection.cursor()

cursor.execute("""
    SELECT activity_name, step_number, image_path, audio_path
    FROM activity_steps
    ORDER BY activity_id, step_number
""")

records = cursor.fetchall()

print("LITTLE STEPS — FILE CHECK")
print("-" * 45)

for activity, step, image, audio in records:
    image_ok = image and os.path.isfile(image)
    audio_ok = audio and os.path.isfile(audio)

    status = "✓" if image_ok and audio_ok else "⚠"

    print(f"{status} {activity} — Step {step}")
    if not image_ok:
        print("   Missing image:", image)
    if not audio_ok:
        print("   Missing audio:", audio)

print("-" * 45)
print(f"Checked {len(records)} steps.")

In [ ]:
import ipywidgets as widgets
from IPython.display import display

# Accessibility controls
text_size = widgets.Dropdown(
    options=["Small", "Medium", "Large"],
    value="Medium",
    description="Text size:"
)

high_contrast = widgets.Checkbox(
    value=False,
    description="High contrast"
)

simple_mode = widgets.Checkbox(
    value=True,
    description="Simple mode"
)

accessibility_title = widgets.HTML(
    "<h3>⚙️ Accessibility Settings</h3>"
)

display(widgets.VBox([
    accessibility_title,
    text_size,
    high_contrast,
    simple_mode
]))

### Older standalone progress dashboard — superseded

This older interface is preserved in the original uploaded notebook. In this revised copy, use the unified interactive app launcher in the final section instead, to avoid duplicate interfaces.

In [ ]:
import shutil
import os

shutil.copy(
    "little_steps.db",
    os.path.join(database_folder, "little_steps.db")
)

print("Little Steps database saved successfully!")

### Older separate welcome/activity menu — superseded

This older interface is preserved in the original uploaded notebook. In this revised copy, use the unified interactive app launcher in the final section instead, to avoid duplicate interfaces.

In [ ]:

from google.colab import drive
import os
import shutil
import sqlite3
import ipywidgets as widgets
from IPython.display import display, HTML, Image, Audio, clear_output

# Reconnect Google Drive
drive.mount('/content/drive')

# Restore project folders
database_folder = "/content/drive/MyDrive/LittleSteps"
image_folder = os.path.join(database_folder, "images")
audio_folder = os.path.join(database_folder, "audio")

# Restore database from Google Drive
shutil.copy(
    os.path.join(database_folder, "little_steps.db"),
    "little_steps.db"
)

# Reconnect SQLite
connection = sqlite3.connect("little_steps.db")
cursor = connection.cursor()

# Restore activity list
cursor.execute("""
SELECT DISTINCT activity_name
FROM activity_steps
ORDER BY activity_id
""")

activities_list = [row[0] for row in cursor.fetchall()]

print("Little Steps successfully reconnected!")
print("Activities:", activities_list)
print("Images folder exists:", os.path.exists(image_folder))
print("Audio folder exists:", os.path.exists(audio_folder))


In [ ]:

import os
import sqlite3

cursor.execute("""
SELECT activity_name, step_number, instruction,
       image_path, audio_path
FROM activity_steps
ORDER BY activity_id, step_number
""")

records = cursor.fetchall()

print("🌈 LITTLE STEPS — FINAL TEST")
print("=" * 45)

passed = 0
failed = 0

for activity, step, instruction, image, audio in records:

    image_ok = bool(image and os.path.isfile(image))
    audio_ok = bool(audio and os.path.isfile(audio))

    if image_ok and audio_ok:
        print(f"✅ {activity} — Step {step}")
        passed += 1
    else:
        print(f"❌ {activity} — Step {step}")
        if not image_ok:
            print("   Missing image")
        if not audio_ok:
            print("   Missing audio")
        failed += 1

print("=" * 45)
print("Total steps:", len(records))
print("Passed:", passed)
print("Need fixing:", failed)

if failed == 0:
    print("\n🎉 ALL FILES ARE READY!")


In [ ]:

# LITTLE STEPS — REAL DATA COMPONENT

!pip -q install ucimlrepo

import pandas as pd
from ucimlrepo import fetch_ucirepo

# Download real dataset from UCI
adl_dataset = fetch_ucirepo(id=271)

# Separate input features and activity labels
X = adl_dataset.data.features
y = adl_dataset.data.targets

print("LITTLE STEPS — DATA EXPLORATION")
print("=" * 40)

print("Dataset shape:", X.shape)
print("Number of labels:", len(y))

print("\nFirst 5 sensor records:")
display(X.head())

print("\nActivity labels:")
display(y.head())

print("\nDataset information:")
print(adl_dataset.metadata)


In [ ]:

import requests
import zipfile
import os

# Download the original UCI dataset
url = "https://archive.ics.uci.edu/static/public/271/activities%2Bof%2Bdaily%2Bliving%2Badls%2Brecognition%2Busing%2Bbinary%2Bsensors.zip"

response = requests.get(url)
response.raise_for_status()

zip_path = "/content/adl_dataset.zip"

with open(zip_path, "wb") as file:
    file.write(response.content)

# Extract dataset
extract_folder = "/content/adl_dataset"
os.makedirs(extract_folder, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as archive:
    archive.extractall(extract_folder)

print("Dataset downloaded successfully!")

# Display available files
for root, folders, files in os.walk(extract_folder):
    for filename in files:
        print(os.path.join(root, filename))


In [ ]:
readme_path = None

for root, folders, files in os.walk(extract_folder):
    for filename in files:
        if filename.lower() == "readme.txt":
            readme_path = os.path.join(root, filename)

if readme_path:
    with open(readme_path, "r", errors="replace") as file:
        print(file.read())
else:
    print("README not found. Check the file list above.")

In [ ]:

import os

for filename in [
    "OrdonezA_Sensors.txt",
    "OrdonezA_ADLs.txt",
    "OrdonezB_Sensors.txt",
    "OrdonezB_ADLs.txt"
]:
    filepath = None

    for root, dirs, files in os.walk(extract_folder):
        if filename in files:
            filepath = os.path.join(root, filename)
            break

    print("\n" + "=" * 60)
    print(filename)

    if filepath:
        with open(filepath, "r", encoding="latin-1") as file:
            for i, line in enumerate(file):
                if i == 8:
                    break
                print(repr(line.rstrip()))
    else:
        print("FILE NOT FOUND")


In [ ]:

import os

filenames = [
    "OrdonezA_Sensors.txt",
    "OrdonezA_ADLs.txt",
    "OrdonezB_Sensors.txt",
    "OrdonezB_ADLs.txt"
]

for filename in filenames:
    path = find_file(filename)

    print("\n" + "=" * 55)
    print(filename)

    with open(path, "r", encoding="latin-1") as file:
        for i, line in enumerate(file):
            if i == 12:
                break
            print(repr(line.rstrip()))


In [ ]:

import pandas as pd
import re
import os

def load_clean_file(filename, expected_columns, column_names):

    path = find_file(filename)
    rows = []

    with open(path, "r", encoding="latin-1") as file:
        for line in file:
            line = line.strip()

            if not line or line.startswith("---"):
                continue

            parts = [
                value.strip()
                for value in re.split(r"\t+", line)
            ]

            if len(parts) != expected_columns:
                continue

            rows.append(parts)

    return pd.DataFrame(rows, columns=column_names)


# Load both users' sensor records
sensor_columns = [
    "start_time", "end_time", "location", "sensor_type", "place"
]

activity_columns = [
    "start_time", "end_time", "activity"
]

sensors_A = load_clean_file(
    "OrdonezA_Sensors.txt", 5, sensor_columns
)

sensors_B = load_clean_file(
    "OrdonezB_Sensors.txt", 5, sensor_columns
)

activities_A = load_clean_file(
    "OrdonezA_ADLs.txt", 3, activity_columns
)

activities_B = load_clean_file(
    "OrdonezB_ADLs.txt", 3, activity_columns
)

# Identify the user
sensors_A["user"] = "A"
sensors_B["user"] = "B"

activities_A["user"] = "A"
activities_B["user"] = "B"

# Combine records
sensors = pd.concat(
    [sensors_A, sensors_B],
    ignore_index=True
)

activities = pd.concat(
    [activities_A, activities_B],
    ignore_index=True
)

# Convert timestamps
for column in ["start_time", "end_time"]:
    sensors[column] = pd.to_datetime(
        sensors[column], errors="coerce"
    )
    activities[column] = pd.to_datetime(
        activities[column], errors="coerce"
    )

# Remove invalid timestamps
sensors = sensors.dropna(
    subset=["start_time", "end_time"]
)

activities = activities.dropna(
    subset=["start_time", "end_time"]
)

print("DATA CLEANING COMPLETE")
print("=" * 40)
print("Sensor records:", len(sensors))
print("Activity records:", len(activities))

print("\nSensor sample:")
display(sensors.head())

print("\nActivity sample:")
display(activities.head())

print("\nAvailable activities:")
print(activities["activity"].value_counts())


In [ ]:

sensors["event_midpoint"] = (
    sensors["start_time"]
    + (sensors["end_time"] - sensors["start_time"]) / 2
)

matched_records = []

for user in ["A", "B"]:

    user_sensors = sensors[sensors["user"] == user]
    user_activities = activities[activities["user"] == user]

    for _, event in user_sensors.iterrows():

        matching = user_activities[
            (user_activities["start_time"] <= event["event_midpoint"]) &
            (user_activities["end_time"] >= event["event_midpoint"])
        ]

        if not matching.empty:
            record = event.to_dict()
            record["activity"] = matching.iloc[0]["activity"]
            matched_records.append(record)

ml_data = pd.DataFrame(matched_records)

print("MATCHING COMPLETE")
print("Matched sensor events:", len(ml_data))
print("\nActivity distribution:")
display(ml_data["activity"].value_counts().to_frame())


In [ ]:

import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt

# Create model features
model_data = ml_data.copy()

model_data["duration_seconds"] = (
    model_data["end_time"] - model_data["start_time"]
).dt.total_seconds().clip(lower=0)

model_data["hour"] = model_data["start_time"].dt.hour

features = [
    "location",
    "sensor_type",
    "place",
    "duration_seconds",
    "hour"
]

target = "activity"

# Train on User A; test on User B
train = model_data[model_data["user"] == "A"].copy()
test = model_data[model_data["user"] == "B"].copy()

if train.empty or test.empty:
    raise ValueError("Missing training or testing records.")

X_train = train[features]
y_train = train[target]

X_test = test[features]
y_test = test[target]

# Prepare numerical and categorical features
categorical_features = ["location", "sensor_type", "place"]
numeric_features = ["duration_seconds", "hour"]

preprocessor = ColumnTransformer([
    ("categorical",
     OneHotEncoder(handle_unknown="ignore"),
     categorical_features),

    ("numeric",
     Pipeline([
         ("imputer", SimpleImputer(strategy="median")),
         ("scaler", StandardScaler())
     ]),
     numeric_features)
])

# Build and train model
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=2000,
        class_weight="balanced"
    ))
])

model.fit(X_train, y_train)

# Make predictions on unseen User B data
predictions = model.predict(X_test)

accuracy = accuracy_score(y_test, predictions)

print("LITTLE STEPS — ACTIVITY RECOGNITION MODEL")
print("=" * 50)
print("Training records:", len(X_train))
print("Testing records:", len(X_test))
print("Training user: A")
print("Testing user: B")
print(f"\nTest accuracy: {accuracy:.2%}")

print("\nClassification report:")
print(classification_report(
    y_test,
    predictions,
    zero_division=0
))

# Plot confusion matrix
ConfusionMatrixDisplay.from_predictions(
    y_test,
    predictions,
    xticks_rotation=90,
    zero_division=0
)

plt.title("Activity Recognition — User B Test Results")
plt.tight_layout()
plt.show()


In [ ]:

# Match sensor events with their labelled activities

sensors["event_midpoint"] = (
    sensors["start_time"]
    + (sensors["end_time"] - sensors["start_time"]) / 2
)

matched_records = []

for user in ["A", "B"]:
    user_sensors = sensors[sensors["user"] == user]
    user_activities = activities[activities["user"] == user]

    for _, event in user_sensors.iterrows():
        matching = user_activities[
            (user_activities["start_time"] <= event["event_midpoint"]) &
            (user_activities["end_time"] >= event["event_midpoint"])
        ]

        if not matching.empty:
            record = event.to_dict()
            record["activity"] = matching.iloc[0]["activity"]
            matched_records.append(record)

ml_data = pd.DataFrame(matched_records)

print("Matching complete!")
print("Total records:", len(ml_data))
print(ml_data["activity"].value_counts())


In [ ]:

# LITTLE STEPS — ACTIVITY RECOGNITION MODEL

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# 1. Match sensor events with activity labels
sensors["event_midpoint"] = (
    sensors["start_time"]
    + (sensors["end_time"] - sensors["start_time"]) / 2
)

matched_records = []

for user in ["A", "B"]:
    user_sensors = sensors[sensors["user"] == user]
    user_activities = activities[activities["user"] == user]

    for _, event in user_sensors.iterrows():
        matching = user_activities[
            (user_activities["start_time"] <= event["event_midpoint"]) &
            (user_activities["end_time"] >= event["event_midpoint"])
        ]

        if not matching.empty:
            record = event.to_dict()
            record["activity"] = matching.iloc[0]["activity"]
            matched_records.append(record)

ml_data = pd.DataFrame(matched_records)

if ml_data.empty:
    raise ValueError("No matching records found. Check your data-loading cell.")

print("Matched records:", len(ml_data))

# 2. Prepare model features
model_data = ml_data.copy()
model_data["duration_seconds"] = (
    model_data["end_time"] - model_data["start_time"]
).dt.total_seconds().clip(lower=0)

model_data["hour"] = model_data["start_time"].dt.hour

features = [
    "location", "sensor_type", "place",
    "duration_seconds", "hour"
]

target = "activity"

# 3. Split by user
train = model_data[model_data["user"] == "A"]
test = model_data[model_data["user"] == "B"]

if train.empty or test.empty:
    raise ValueError("Training or testing data is empty.")

X_train = train[features]
y_train = train[target]
X_test = test[features]
y_test = test[target]

# 4. Build and train the model
categorical = ["location", "sensor_type", "place"]
numerical = ["duration_seconds", "hour"]

preprocessor = ColumnTransformer([
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical),

    ("numerical", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), numerical)
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=2000,
        class_weight="balanced"
    ))
])

model.fit(X_train, y_train)

# 5. Evaluate on User B's data
predictions = model.predict(X_test)

print("\nMODEL EVALUATION")
print("Accuracy:", round(accuracy_score(y_test, predictions) * 100, 2), "%")
print("\nClassification Report:")
print(classification_report(y_test, predictions, zero_division=0))


In [ ]:

import os
import joblib
from sklearn.metrics import classification_report, accuracy_score

# Save the trained model
model_path = os.path.join(
    database_folder, "activity_recognition_model.joblib"
)
joblib.dump(model, model_path)

# Save evaluation results
report = classification_report(
    y_test, predictions, zero_division=0
)

report_path = os.path.join(
    database_folder, "model_evaluation.txt"
)

with open(report_path, "w") as file:
    file.write("Little Steps - Activity Recognition Model\n")
    file.write(f"Test accuracy: {accuracy_score(y_test, predictions) * 100:.2f}%\n\n")
    file.write(report)

print("MODEL SAVED!")
print("Model:", model_path)
print("Evaluation:", report_path)


In [ ]:

import os
import joblib
from sklearn.metrics import accuracy_score, classification_report

# Make sure the Drive project folder exists
os.makedirs(database_folder, exist_ok=True)

# Save trained model
model_path = os.path.join(
    database_folder, "activity_recognition_model.joblib"
)
joblib.dump(model, model_path)

# Save evaluation results
report_path = os.path.join(
    database_folder, "model_evaluation.txt"
)

with open(report_path, "w") as file:
    file.write("Little Steps - ML Evaluation\n")
    file.write(
        f"Accuracy: {accuracy_score(y_test, predictions) * 100:.2f}%\n\n"
    )
    file.write(classification_report(
        y_test, predictions, zero_division=0
    ))

print("Model and evaluation saved successfully!")
print(model_path)
print(report_path)



### Old database rebuild cell — disabled for safety

This legacy repair cell deletes and reinserts activity rows. The database and media paths have already been restored, so do not run this cell. The final app launcher safely checks the existing records without wiping them.

### Old database-lock recovery rebuild cell — disabled for safety

This legacy repair cell deletes and reinserts activity rows. The database and media paths have already been restored, so do not run this cell. The final app launcher safely checks the existing records without wiping them.

In [ ]:

import os

print("IMAGE FILES:", len(os.listdir(image_folder)))
print("AUDIO FILES:", len(os.listdir(audio_folder)))

print("\nSample images:")
print(os.listdir(image_folder)[:10])

print("\nSample audio:")
print(os.listdir(audio_folder)[:10])


In [ ]:

import os

database_folder = "/content/drive/MyDrive/LittleSteps"
image_folder = os.path.join(database_folder, "images")
audio_folder = os.path.join(database_folder, "audio")

print("Project folder exists:", os.path.exists(database_folder))
print("Images folder exists:", os.path.exists(image_folder))
print("Audio folder exists:", os.path.exists(audio_folder))

if os.path.exists(image_folder):
    print("Image files:", len(os.listdir(image_folder)))

if os.path.exists(audio_folder):
    print("Audio files:", len(os.listdir(audio_folder)))


In [ ]:

print("=== IMAGE FILES ===")
for filename in sorted(os.listdir(image_folder)):
    print(filename)

print("\n=== AUDIO FILES ===")
for filename in sorted(os.listdir(audio_folder)):
    print(filename)


In [ ]:

import os
import sqlite3

connection = sqlite3.connect("little_steps.db")
cursor = connection.cursor()

# Reconnect every activity step to its image and audio
cursor.execute("SELECT activity_id, activity_name, step_number FROM activity_steps")
rows = cursor.fetchall()

for activity_id, activity_name, step_number in rows:

    prefix = activity_name.lower().replace(" ", "_")

    image_path = os.path.join(
        image_folder,
        f"{prefix}_step_{step_number}.png"
    )

    audio_path = os.path.join(
        audio_folder,
        f"{prefix}_step_{step_number}.mp3"
    )

    cursor.execute("""
        UPDATE activity_steps
        SET image_path = ?, audio_path = ?
        WHERE activity_id = ? AND step_number = ?
    """, (
        image_path,
        audio_path,
        activity_id,
        step_number
    ))

connection.commit()

# Verify all connections
cursor.execute("""
SELECT activity_name, step_number, image_path, audio_path
FROM activity_steps
ORDER BY activity_id, step_number
""")

missing = []

for activity, step, image, audio in cursor.fetchall():
    if not os.path.exists(image) or not os.path.exists(audio):
        missing.append((activity, step))

print("Total steps:", len(rows))
print("Missing image/audio connections:", len(missing))
print("All paths restored!" if not missing else missing)

connection.close()


In [ ]:
import shutil
import sqlite3
import os

# Save repaired database to Google Drive
shutil.copy(
    "little_steps.db",
    os.path.join(database_folder, "little_steps.db")
)

# Check database structure
connection = sqlite3.connect("little_steps.db")
cursor = connection.cursor()

cursor.execute("PRAGMA table_info(activity_steps)")
columns = [row[1] for row in cursor.fetchall()]

cursor.execute("SELECT COUNT(*) FROM activity_steps")
total = cursor.fetchone()[0]

print("Database saved to Drive!")
print("Total steps:", total)
print("Database columns:", columns)

connection.close()

In [ ]:
import sqlite3

connection = sqlite3.connect("little_steps.db")
cursor = connection.cursor()

cursor.execute("""
SELECT activity_name
FROM activity_steps
GROUP BY activity_id, activity_name
ORDER BY activity_id
""")

activities_list = [row[0] for row in cursor.fetchall()]

print("Activities ready:")
for activity in activities_list:
    print("✓", activity)

print("Total activities:", len(activities_list))

In [ ]:
import os
import sqlite3
from IPython.display import display, Image, Audio

connection = sqlite3.connect("little_steps.db")
cursor = connection.cursor()

cursor.execute("""
SELECT activity_name, step_number, instruction, image_path, audio_path
FROM activity_steps
ORDER BY activity_id, step_number
""")

rows = cursor.fetchall()

for activity, step, instruction, image_path, audio_path in rows:
    print(f"\n{activity} — Step {step}")
    print(instruction)

    if os.path.exists(image_path):
        display(Image(filename=image_path, width=300))
    else:
        print("IMAGE MISSING")

    if os.path.exists(audio_path):
        display(Audio(filename=audio_path))
    else:
        print("AUDIO MISSING")

connection.close()

# Little Steps — Project Overview

## Problem
Children with disabilities may find everyday activities difficult to learn when instructions are complicated, overwhelming, or inaccessible.

## Solution
Little Steps is designed to support children with disabilities and children who benefit from visual, step-by-step guidance when learning everyday activities.

## Main Features
- Step-by-step learning activities
- 3D visual illustrations
- Spoken instructions
- Previous and Next navigation
- Progress tracking
- Adjustable text size
- High-contrast mode
- Simplified interface

## Activities
1. Washing Hands
2. Brushing Teeth
3. Tying Shoes

## Machine Learning Component
We explored activity recognition using the UCI Activities of Daily Living dataset. A Logistic Regression model was trained using sensor-event information.

The model achieved **58.25% accuracy** on a test set from the second user in the dataset.

This is an experimental component and is separate from the instructional features of Little Steps.

## Limitations
- The ML dataset contains sensor data from only two homes.
- Some activities had very few examples.
- The model does not recognize children performing tasks.
- The learning interface has only three activities in this prototype.

## Future Improvements
- Add more everyday activities.
- Include customizable instructions.
- Improve accessibility and personalization.
- Evaluate the application with appropriate users and feedback.

## FINAL — Launch Little Steps Interactive App

Run the code cell below after your Google Drive project folder contains:
- `LittleSteps/little_steps.db`
- `LittleSteps/images/`
- `LittleSteps/audio/`

The app opens with **picture-based activity cards**. Select a card, view each illustrated step, replay its spoken instruction, mark the step complete, and track progress. Accessibility controls include text size, high contrast, and simple instructions.


In [ ]:
# LITTLE STEPS — INTERACTIVE APP (SAFE LAUNCH)
# Run this cell after connecting Google Drive. It does NOT delete or rebuild your data.

import os
import html
import sqlite3
import shutil
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML, Image as IPyImage

# 1. Connect to the existing Google Drive project
from google.colab import drive
drive.mount("/content/drive")

PROJECT_FOLDER = "/content/drive/MyDrive/LittleSteps"
IMAGE_FOLDER = os.path.join(PROJECT_FOLDER, "images")
AUDIO_FOLDER = os.path.join(PROJECT_FOLDER, "audio")
DRIVE_DB = os.path.join(PROJECT_FOLDER, "little_steps.db")
LOCAL_DB = "/content/little_steps.db"

if not os.path.isfile(DRIVE_DB):
    raise FileNotFoundError(
        "I couldn't find LittleSteps/little_steps.db in Google Drive. "
        "Please check that the LittleSteps folder is in MyDrive."
    )

# Use a local working copy, preserving the Drive backup
shutil.copy2(DRIVE_DB, LOCAL_DB)

db = sqlite3.connect(LOCAL_DB, timeout=30)
cur = db.cursor()

def save_database_backup():
    """Safely save the current database, including progress, back to Drive."""
    with sqlite3.connect(DRIVE_DB, timeout=30) as drive_connection:
        db.backup(drive_connection)

# Ensure the progress column exists without deleting any rows
cur.execute("PRAGMA table_info(activity_steps)")
db_columns = [row[1] for row in cur.fetchall()]
if "completed" not in db_columns:
    cur.execute("ALTER TABLE activity_steps ADD COLUMN completed INTEGER DEFAULT 0")
    db.commit()

# Restore file paths only if an existing path is missing or invalid
cur.execute("""
    SELECT activity_id, activity_name, step_number, image_path, audio_path
    FROM activity_steps
    ORDER BY activity_id, step_number
""")
db_rows = cur.fetchall()

for activity_id, activity_name, step_number, image_path, audio_path in db_rows:
    prefix = activity_name.lower().replace(" ", "_")
    expected_image = os.path.join(IMAGE_FOLDER, f"{prefix}_step_{step_number}.png")
    expected_audio = os.path.join(AUDIO_FOLDER, f"{prefix}_step_{step_number}.mp3")

    new_image = image_path if image_path and os.path.isfile(image_path) else expected_image
    new_audio = audio_path if audio_path and os.path.isfile(audio_path) else expected_audio

    if new_image != image_path or new_audio != audio_path:
        cur.execute("""
            UPDATE activity_steps
            SET image_path = ?, audio_path = ?
            WHERE activity_id = ? AND step_number = ?
        """, (new_image, new_audio, activity_id, step_number))

db.commit()
save_database_backup()

cur.execute("""
    SELECT DISTINCT activity_id, activity_name
    FROM activity_steps
    ORDER BY activity_id
""")
ACTIVITIES = cur.fetchall()

if not ACTIVITIES:
    raise ValueError("The activity database is empty. No activity cards can be shown.")

# 2. Build the interface
current_activity_id = None
current_activity_name = None
current_step_number = 1

main_title = widgets.HTML("""
<div style="background:linear-gradient(120deg,#287C78,#5C9EBD,#9274B8);
padding:26px 18px;border-radius:22px;text-align:center;color:white;
box-shadow:0 5px 16px rgba(30,70,90,.15);">
  <div style="font-size:38px;">🌈</div>
  <h1 style="margin:4px 0;font-size:34px;">Little Steps</h1>
  <p style="font-size:17px;margin:0;">Every big achievement starts with a little step!</p>
</div>
""")

intro = widgets.HTML("""
<div style="padding:12px 4px;text-align:center;">
  <h2 style="margin-bottom:4px;">What would you like to learn today?</h2>
  <p style="margin-top:0;color:#64748b;">Choose a picture to get started.</p>
</div>
""")

text_size = widgets.Dropdown(
    options=[("Small", "Small"), ("Medium", "Medium"), ("Large", "Large")],
    value="Medium",
    description="Text size:"
)
high_contrast = widgets.Checkbox(value=False, description="High contrast")
simple_mode = widgets.Checkbox(value=True, description="Simple instructions")

settings_title = widgets.HTML("<h3 style='margin:10px 0 4px;'>⚙️ Accessibility settings</h3>")
cards_area = widgets.Output()
step_area = widgets.Output()
dashboard_area = widgets.Output()

back_button = widgets.Button(
    description="← Choose another activity",
    button_style="",
    layout=widgets.Layout(width="230px", height="42px")
)
previous_button = widgets.Button(
    description="← Previous",
    layout=widgets.Layout(width="140px", height="44px")
)
next_button = widgets.Button(
    description="Next step →",
    button_style="success",
    layout=widgets.Layout(width="150px", height="44px")
)
complete_button = widgets.Button(
    description="☆ I completed this step!",
    button_style="info",
    layout=widgets.Layout(width="230px", height="44px")
)
replay_button = widgets.Button(
    description="🔊 Hear instruction again",
    layout=widgets.Layout(width="230px", height="44px")
)
step_controls = widgets.VBox([
    widgets.HBox([previous_button, next_button]),
    widgets.HBox([complete_button, replay_button]),
    back_button
], layout=widgets.Layout(align_items="center"))

step_progress = widgets.IntProgress(
    value=0, min=0, max=100, description="Activity:",
    bar_style="success", layout=widgets.Layout(width="100%")
)

def _get_step():
    cur.execute("""
        SELECT instruction, image_path, audio_path, completed
        FROM activity_steps
        WHERE activity_id = ? AND step_number = ?
    """, (current_activity_id, current_step_number))
    return cur.fetchone()

def _update_dashboard():
    with dashboard_area:
        clear_output(wait=True)
        cur.execute("""
            SELECT activity_name, COUNT(*), COALESCE(SUM(completed), 0)
            FROM activity_steps
            GROUP BY activity_id, activity_name
            ORDER BY activity_id
        """)
        rows = cur.fetchall()
        total_steps = sum(row[1] for row in rows)
        total_done = sum(row[2] for row in rows)

        cards = []
        for name, total, done in rows:
            pct = int(done * 100 / total) if total else 0
            cards.append(f"""
            <div style="background:#f1f7fb;border-radius:14px;padding:13px;margin:9px 0;">
              <div style="font-weight:700;font-size:16px;">{html.escape(name)}</div>
              <div style="font-size:13px;margin:5px 0;">{done} of {total} steps completed</div>
              <div style="height:12px;background:#dce7ee;border-radius:20px;overflow:hidden;">
                <div style="height:12px;width:{pct}%;background:#399b79;border-radius:20px;"></div>
              </div>
              <div style="text-align:right;font-weight:700;font-size:13px;margin-top:4px;">{pct}%</div>
            </div>
            """)
        overall = int(total_done * 100 / total_steps) if total_steps else 0
        display(HTML(
            "<h3>⭐ My Little Steps progress</h3>"
            + "".join(cards)
            + f"<h3>Overall: {total_done}/{total_steps} steps ({overall}%)</h3>"
        ))

def _show_activity_cards():
    with cards_area:
        clear_output(wait=True)
        activity_cards = []

        for activity_id, activity_name in ACTIVITIES:
            slug = activity_name.lower().replace(" ", "_")
            cover_path = os.path.join(IMAGE_FOLDER, f"{slug}_cover.png")
            if not os.path.isfile(cover_path):
                cover_path = os.path.join(IMAGE_FOLDER, f"{slug}_step_1.png")

            image_widget = None
            if os.path.isfile(cover_path):
                with open(cover_path, "rb") as image_file:
                    image_widget = widgets.Image(
                        value=image_file.read(),
                        format="png",
                        layout=widgets.Layout(
                            width="190px", height="145px",
                            object_fit="contain",
                            border="1px solid #e2e8f0"
                        )
                    )
            else:
                image_widget = widgets.HTML(
                    "<div style='height:145px;width:190px;display:flex;"
                    "align-items:center;justify-content:center;background:#eef4f7;"
                    "border-radius:12px;font-size:40px;'>📘</div>"
                )

            choose_button = widgets.Button(
                description="Choose this activity",
                button_style="info",
                layout=widgets.Layout(width="190px", height="42px")
            )
            choose_button.on_click(
                lambda button, aid=activity_id, name=activity_name: open_activity(aid, name)
            )

            card = widgets.VBox(
                [
                    image_widget,
                    widgets.HTML(
                        f"<div style='font-weight:700;text-align:center;"
                        f"font-size:15px;padding:5px 0;'>{html.escape(activity_name)}</div>"
                    ),
                    choose_button
                ],
                layout=widgets.Layout(
                    width="215px", padding="10px", margin="6px",
                    border="1px solid #dbe6ec",
                    align_items="center"
                )
            )
            activity_cards.append(card)

        display(widgets.Box(
            activity_cards,
            layout=widgets.Layout(
                display="flex", flex_flow="row wrap",
                justify_content="center", align_items="stretch"
            )
        ))

def _render_step():
    if current_activity_id is None:
        return

    result = _get_step()
    with step_area:
        clear_output(wait=True)

        if not result:
            display(HTML("<p>We couldn't find this step. Please choose the activity again.</p>"))
            return

        instruction, image_path, audio_path, completed = result

        cur.execute("""
            SELECT COUNT(*), COALESCE(SUM(completed), 0)
            FROM activity_steps
            WHERE activity_id = ?
        """, (current_activity_id,))
        total, done = cur.fetchone()
        step_progress.value = int(done * 100 / total) if total else 0

        font_sizes = {"Small": "20px", "Medium": "27px", "Large": "36px"}
        font_size = font_sizes[text_size.value]
        bg = "#000000" if high_contrast.value else "#f3f8fc"
        fg = "#ffff00" if high_contrast.value else "#203b53"

        step_label = ""
        if not simple_mode.value:
            step_label = f"<div style='font-size:15px;margin-bottom:10px;'>Step {current_step_number} of {total}</div>"

        display(HTML(f"""
        <div style="background:{bg};color:{fg};padding:22px;border-radius:18px;
                    text-align:center;border:1px solid #dbe5eb;">
          <div style="font-size:14px;letter-spacing:1px;text-transform:uppercase;">
            {html.escape(current_activity_name)}
          </div>
          {step_label}
          <div style="font-size:{font_size};font-weight:700;line-height:1.35;margin-top:8px;">
            {html.escape(instruction)}
          </div>
        </div>
        """))

        if image_path and os.path.isfile(image_path):
            # Display at a fixed width while preserving the file's true aspect ratio.
            # This avoids squeezing or clipping tall/wide illustrations in a fixed box.
            display(IPyImage(filename=image_path, width=520))
        else:
            display(HTML("<p>🖼️ Picture not available for this step.</p>"))

        if audio_path and os.path.isfile(audio_path):
            from IPython.display import Audio
            display(Audio(filename=audio_path))
        else:
            display(HTML("<p>🔈 Audio is not available for this step.</p>"))

        complete_button.description = (
            "✓ Completed — tap to undo" if completed else "☆ I completed this step!"
        )
        complete_button.button_style = "success" if completed else "info"
        previous_button.disabled = current_step_number <= 1
        next_button.description = "Finish activity ✓" if current_step_number >= total else "Next step →"

def open_activity(activity_id, activity_name):
    global current_activity_id, current_activity_name, current_step_number
    current_activity_id = activity_id
    current_activity_name = activity_name
    current_step_number = 1
    cards_area.layout.display = "none"
    step_area.layout.display = "block"
    step_controls.layout.display = "flex"
    _render_step()
    _update_dashboard()

def back_to_activities(_=None):
    global current_activity_id, current_activity_name, current_step_number
    current_activity_id = None
    current_activity_name = None
    current_step_number = 1
    step_area.clear_output()
    step_area.layout.display = "none"
    step_controls.layout.display = "none"
    cards_area.layout.display = "block"
    _show_activity_cards()
    _update_dashboard()

def move_step(direction):
    global current_step_number
    if current_activity_id is None:
        return
    cur.execute(
        "SELECT COUNT(*) FROM activity_steps WHERE activity_id = ?",
        (current_activity_id,)
    )
    total = cur.fetchone()[0]
    current_step_number = max(1, min(total, current_step_number + direction))
    _render_step()

def toggle_completed(_):
    if current_activity_id is None:
        return
    result = _get_step()
    if not result:
        return
    completed = int(not bool(result[3]))
    cur.execute("""
        UPDATE activity_steps
        SET completed = ?
        WHERE activity_id = ? AND step_number = ?
    """, (completed, current_activity_id, current_step_number))
    db.commit()
    save_database_backup()
    _render_step()
    _update_dashboard()

def replay_audio(_):
    result = _get_step()
    if result and result[2] and os.path.isfile(result[2]):
        from IPython.display import Audio
        with step_area:
            display(HTML("<p><b>🔊 Playing the instruction again:</b></p>"))
            display(Audio(filename=result[2]))
    else:
        with step_area:
            display(HTML("<p>Audio is not available for this step.</p>"))

def accessibility_changed(change):
    if change.get("name") == "value" and current_activity_id is not None:
        _render_step()

previous_button.on_click(lambda _: move_step(-1))
next_button.on_click(lambda _: move_step(1))
complete_button.on_click(toggle_completed)
replay_button.on_click(replay_audio)
back_button.on_click(back_to_activities)

text_size.observe(accessibility_changed, names="value")
high_contrast.observe(accessibility_changed, names="value")
simple_mode.observe(accessibility_changed, names="value")

# Initial display
display(widgets.VBox([
    main_title,
    settings_title,
    widgets.HBox([text_size, high_contrast, simple_mode]),
    intro,
    cards_area,
    step_area,
    step_controls,
    widgets.HTML("<hr><h3>🌟 Your progress</h3>"),
    dashboard_area
]))

step_area.layout.display = "none"
step_controls.layout.display = "none"
_show_activity_cards()
_update_dashboard()

print("Little Steps is ready! Choose an activity picture to begin.")
